# Trigger Airflow DAGs from a notebook

Needs `make airflow`. The notebook asks Airflow to run a DAG; the DAG's tasks submit the
Spark jobs to Kyuubi. So the chain is notebook -> Airflow REST -> Airflow task -> Kyuubi
REST -> spark-submit. Airflow's UI: http://localhost:8085.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
lab.airflow_dags()

## One job, any job
`kyuubi_run_any_job` takes the job name and optional args/conf.

In [ ]:
state = lab.airflow_trigger("kyuubi_run_any_job", conf={"job": "py-wordcount"}, wait=True)
state

## A whole pipeline
SQL-only reporting over Kyuubi JDBC, then a look at its tasks.

In [ ]:
run_id = lab.airflow_trigger("kyuubi_sql_reporting")
print(lab.airflow_run_state("kyuubi_sql_reporting", run_id))

In [ ]:
import time
while lab.airflow_run_state("kyuubi_sql_reporting", run_id) in ("queued", "running"):
    time.sleep(10)
lab.airflow_tasks("kyuubi_sql_reporting", run_id)

In [ ]:
lab.pg("SELECT * FROM report_region_month ORDER BY month, region LIMIT 10")

## Kafka pipeline with a parameter

In [ ]:
lab.airflow_trigger("kafka_streaming_pipeline", conf={"orders": 25})